# FlyRank ML Internship Capstone — Position-Aware Search Content Refresh Opportunity Scoring

**Lane 4 — CTR / Engagement Opportunity Scoring**  
**Author:** Arya Vengurlekar  
**Target Notebook:** `work/notebooks/capstone.ipynb`  
**Repository:** [GitHub - AryaVengurlekar03/Machine-Learning](https://github.com/AryaVengurlekar03/Machine-Learning)  
**Data Credit:** Built on the [FlyRank](https://flyrank.ai) ML Internship dataset


## 1. Question

### Research Question
Can a position-aware machine learning model identify organic search content pieces with significant CTR underperformance relative to position-tier peers, and how does its ranking performance compare with a fixed heuristic baseline under an unseen-client validation design?

### Decision Supported & Cost of a Wrong Call
- **Decision Supported:** Editorial triage for content refreshes — prioritizing which underperforming organic search pages should undergo title/meta tag rewrites or content depth updates.
- **Cost of a Wrong Call:** Wasting finite human editorial capacity (10–15 minutes per page) rewriting meta titles on pages facing structural SERP instant-answer features, or disrupting existing high-performing traffic-generating assets.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load starter dataset
csv_path = '../../data/raw/content_refresh_anonymized.csv' if os.path.exists('../../data/raw/content_refresh_anonymized.csv') else 'data/raw/content_refresh_anonymized.csv'
df_raw = pd.read_csv(csv_path)
print(f'[DATA LOADED] Raw dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns across {df_raw["client_id"].nunique()} clients')


[DATA LOADED] Raw dataset shape: 30,000 rows x 44 columns across 32 clients


## 2. Data

### Dataset & Exclusions
- **Release & Source:** `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized content items across 31 client domains).
- **Date Window:** Trailing 90-day search and web analytics performance snapshot.
- **Filtering & Exclusions:**
  - Excluded 1,205 rows with `avg_position == 0` (no Google Search Console rank data), leaving $N=28,795$ valid pages.
  - Excluded target-derived trend columns (`is_declining_label`, `trend_direction`, `trend_pct`) to prevent target leakage.
  - Excluded pseudonymized identifiers (`content_id`, `client_id`) from feature inputs (used strictly for grouping/splitting).
- **Public-Safety Guarantee:** No private client names, raw query strings, or sensitive API credentials exist in this dataset or output queue.

In [2]:
# Filter valid pages with search rank data
df = df_raw[df_raw['avg_position'] > 0].copy()
print(f'[FILTERED] Excluded 1,205 zero-rank rows. Valid pages evaluated: {len(df):,}')

# Target computation (matching Week 4 & Week 5 & Week 6)
df['expected_ctr_peer'] = df.groupby(['position_tier', 'main_intent'])['ctr'].transform('median')
df['expected_ctr_peer'] = df['expected_ctr_peer'].fillna(df.groupby('position_tier')['ctr'].transform('median'))
df['ctr_gap'] = (df['expected_ctr_peer'] - df['ctr']).clip(lower=0)
df['missed_clicks'] = (df['ctr_gap'] / 100.0) * df['impressions_90d']
ACTION_THRESHOLD = 10.0
df['is_actionable'] = (df['missed_clicks'] >= ACTION_THRESHOLD).astype(int)

base_rate = df['is_actionable'].mean()
print(f'[DATASET BASE RATE] Actionable threshold (missed_clicks >= 10.0): {base_rate:.4f} ({base_rate*100:.2f}%) across N={len(df):,} valid pages')


[FILTERED] Excluded 1,205 zero-rank rows. Valid pages evaluated: 28,795
[DATASET BASE RATE] Actionable threshold (missed_clicks >= 10.0): 0.0152 (1.52%) across N=28,795 valid pages


## 3. Methodology

### Feature Matrix & Model Architecture
- **Target Proxy:** `missed_clicks = max(0, impressions_90d * (expected_ctr_peer - ctr))` modeled in log space $\log(1 + \text{missed\_clicks})$.
- **Feature Preprocessing:** Added missingness flags (`has_search_volume`, `has_word_count`, `has_scroll_rate`), filled numerical NAs with 0, and one-hot encoded categorical attributes (`position_tier`, `impression_tier`, `content_type`, `main_intent`, `competition_level`, `age_tier`, `freshness_tier`).
- **Model Selection:** `HistGradientBoostingRegressor` (max_iter=100, max_depth=6).
- **Validation Design:**
  1. **Honest Grouped Split by Client (`GroupShuffleSplit`):** 25 train clients / 6 test clients (Unseen Test $N=5,821$). Evaluates generalization to entirely new client sites.
  2. **5-Fold GroupKFold Out-of-Fold (OOF):** Evaluated across all 31 clients ($N=28,795$).
- **Leakage Audit:** 100% strict exclusion of target/sibling variables (`missed_clicks`, `ctr`, `clicks_90d`, `is_declining_label`) from input matrix $X$.

In [3]:
# Feature preparation: missingness flags & numerical filling
df['has_search_volume'] = df['search_volume'].notna().astype(int)
df['has_word_count'] = df['word_count'].notna().astype(int)
df['has_scroll_rate'] = df['scroll_rate'].notna().astype(int)

num_cols = [
    'impressions_90d', 'avg_position', 'search_volume', 'competition', 'cpc',
    'word_count', 'char_count', 'days_since_last_update', 'content_age_days',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'sessions_90d', 'pageviews_90d',
    'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'days_with_impressions',
    'days_with_sessions', 'impressions_last_30d', 'impressions_prev_30d',
    'sessions_last_30d', 'sessions_prev_30d'
]
for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

cat_cols = ['position_tier', 'impression_tier', 'content_type', 'main_intent', 'competition_level', 'age_tier', 'freshness_tier']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

forbidden_leakage_cols = [
    'content_id', 'client_id', 'expected_ctr_peer', 'ctr_gap', 'missed_clicks', 'is_actionable',
    'ctr', 'clicks_90d', 'clicks_last_30d', 'clicks_prev_30d', 'trend_direction', 'trend_pct',
    'is_declining_label', 'provider_used', 'model_used', 'word_count_tier', 'char_count_tier', 'age_tier_order'
]
feature_cols = [c for c in df_encoded.columns if c not in forbidden_leakage_cols and not c.startswith('score')]

# Programmatic Leakage Verification
for col in ['is_declining_label', 'trend_direction', 'trend_pct', 'missed_clicks', 'ctr']:
    assert col not in feature_cols, f'LEAKAGE ERROR: {col} present in features!'
print(f'[LEAKAGE AUDIT PASSED] Total verified safe features: {len(feature_cols)}')

# Execute Client-Grouped Train/Test Split (Week 6 Validation Design)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
tr_g_idx, te_g_idx = next(gss.split(df_encoded, groups=df_encoded['client_id']))
tr_g = df_encoded.iloc[tr_g_idx].copy()
te_g = df_encoded.iloc[te_g_idx].copy()

print(f'[GROUPED SPLIT] Train clients: {tr_g["client_id"].nunique()} ({len(tr_g):,} rows) | Test clients: {te_g["client_id"].nunique()} ({len(te_g):,} rows)')

# Fit HistGradientBoostingRegressor on grouped split
hgb_model = HistGradientBoostingRegressor(max_iter=100, max_depth=6, random_state=RANDOM_SEED)
hgb_model.fit(tr_g[feature_cols], np.log1p(tr_g['missed_clicks']))

te_g['baseline_score'] = te_g['missed_clicks'] # Baseline heuristic rule
te_g['hgb_score'] = np.expm1(hgb_model.predict(te_g[feature_cols]))


[LEAKAGE AUDIT PASSED] Total verified safe features: 47
[GROUPED SPLIT] Train clients: 24 (22,974 rows) | Test clients: 7 (5,821 rows)


## 4. Results (vs baseline)

### Honest Model vs Baseline Comparison & Synthesis
We evaluate the Week 4 Baseline Heuristic Rule vs the Week 6 HistGradientBoosting model under identical unseen-client validation splits (=5,821$, Base Rate = .25\%$).

**Key Findings:**
- **Baseline Lead:** The fixed heuristic baseline outperformed the HistGradientBoosting model across all reported ranking metrics on unseen test clients (P@10: 1.0000 vs 0.9000; P@20: 1.0000 vs 0.8500; P@50: 1.0000 vs 0.7200; P@100: 0.7300 vs 0.5400).
- **Recoverable Clicks:** In the top-10 queue, the heuristic baseline identified 1,206.8 recoverable clicks vs 709.2 for the ML model. In the top-50 queue, the heuristic captured 2,283.7 recoverable clicks vs 1,627.3 for the ML model.
- **ML Utility & Framing:** While the ML model demonstrated useful ranking performance (90.0% P@10, \\times$ lift over the base rate), current evidence does not justify claiming ML superiority over the simpler position-peer heuristic baseline under this evaluation design.

In [4]:
def evaluate_ranking_queue(df_queue, score_col, threshold=ACTION_THRESHOLD, k_list=[10, 20, 50, 100]):
    sorted_df = df_queue.sort_values(score_col, ascending=False).reset_index(drop=True)
    base_r = (sorted_df['missed_clicks'] >= threshold).mean()
    res = {'Base Rate': base_r}
    for k in k_list:
        top_k = sorted_df.head(k)
        res[f'P@{k}'] = (top_k['missed_clicks'] >= threshold).mean()
        res[f'MC@{k}'] = top_k['missed_clicks'].sum()
    return res

m_base = evaluate_ranking_queue(te_g, 'baseline_score')
m_hgb = evaluate_ranking_queue(te_g, 'hgb_score')

comp_df = pd.DataFrame([m_base, m_hgb], index=['Week 4 Baseline Rule', 'HistGradientBoosting (Grouped Split)'])

disp_table = pd.DataFrame({
    'Method': comp_df.index,
    'Base Rate': comp_df['Base Rate'].map(lambda x: f'{x:.4f}'),
    'Precision@10': comp_df['P@10'].map(lambda x: f'{x:.4f} ({x*100:.1f}%)'),
    'Precision@20': comp_df['P@20'].map(lambda x: f'{x:.4f} ({x*100:.1f}%)'),
    'Precision@50': comp_df['P@50'].map(lambda x: f'{x:.4f} ({x*100:.1f}%)'),
    'Precision@100': comp_df['P@100'].map(lambda x: f'{x:.4f} ({x*100:.1f}%)'),
    'Top-10 Recoverable Clicks': comp_df['MC@10'].map(lambda x: f'{x:,.1f}'),
    'Top-50 Recoverable Clicks': comp_df['MC@50'].map(lambda x: f'{x:,.1f}')
})

print('=== UNSEEN TEST CLIENT EVALUATION TABLE (TEST N=5,821) ===')
print(disp_table.to_string(index=False))


=== UNSEEN TEST CLIENT EVALUATION TABLE (TEST N=5,821) ===
                              Method Base Rate    Precision@10    Precision@20    Precision@50  Precision@100 Top-10 Recoverable Clicks Top-50 Recoverable Clicks
                Week 4 Baseline Rule    0.0125 1.0000 (100.0%) 1.0000 (100.0%) 1.0000 (100.0%) 0.7300 (73.0%)                   1,206.8                   2,283.7
HistGradientBoosting (Grouped Split)    0.0125  0.9000 (90.0%)  0.8500 (85.0%)  0.7200 (72.0%) 0.5400 (54.0%)                     709.2                   1,627.3


## 5. Limitations

### Observational Data & Model Boundaries
1. **No Causal Guarantee:** The model identifies pages with historical CTR deficits relative to position peers. It does not guarantee future rank improvements or click recovery.
2. **Zero-Click SERP Features:** Pages ranking for factual/definition terms frequently encounter Google Instant Answer boxes or Knowledge Panels. In these cases, CTR deficits are structural rather than metadata defects.
3. **Survivor Bias:** Cross-sectional snapshot datasets contain surviving pages; deleted underperforming pages are omitted.
4. **ML Model Performance Constraint:** On unseen client domains, the HistGradientBoosting model does not outperform the simpler position-peer heuristic baseline. The heuristic remains the primary operational baseline under this evaluation.
5. **Mandatory Human Review:** Automated publishing or title changes without editorial verification are strictly prohibited.

In [5]:
# Skeptic's Error Analysis: Extract concrete error cases
te_g['abs_error'] = np.abs(te_g['hgb_score'] - te_g['missed_clicks'])
top_errs = te_g.sort_values('abs_error', ascending=False).head(3)

print('=== TOP 3 CONCRETE PREDICTION ERROR EXAMPLES ===')
print(top_errs[['content_id', 'avg_position', 'impressions_90d', 'missed_clicks', 'hgb_score', 'abs_error']].to_string(index=False))


=== TOP 3 CONCRETE PREDICTION ERROR EXAMPLES ===
          content_id  avg_position  impressions_90d  missed_clicks  hgb_score  abs_error
content_c84a0ab98e90           7.8           223271       245.5981   9.151234 236.446866
content_453722754fea           7.6           140079       182.1027  33.550622 148.552078
content_39881853ef0c           7.2           112434       146.1642  46.590392  99.573808


## 6. Ranked recommendations

### Content Refresh Action Playbook
We score all $N=28,795$ valid pages using the trained model to build the prioritized editorial refresh queue with reason codes and action mappings.

In [6]:
# Predict opportunity scores for full dataset
df['opportunity_score'] = np.expm1(hgb_model.predict(df_encoded[feature_cols]))

def assign_playbook_action(row):
    score = row['opportunity_score']
    pos = row['avg_position']
    age = row.get('content_age_days', 0)
    ctr_gap = row['ctr_gap']
    
    if score >= 10.0 and ctr_gap > 0.02:
        return 'CTR_BELOW_POSITION_PEER', 'OPTIMIZE_TITLE_META_SNIPPET'
    elif pos > 10.0 and pos <= 20.0 and row['impressions_90d'] >= 50000:
        return 'STRIKING_DISTANCE_POS_11_20', 'REFRESH_CONTENT_DEPTH_INTENT'
    elif age > 365 and row['impressions_90d'] >= 30000:
        return 'STALE_CONTENT_HIGH_VISIBILITY', 'UPDATE_FRESHNESS_AND_DATES'
    else:
        return 'GENERAL_CTR_REVIEW', 'AUDIT_USER_INTENT_ALIGNMENT'

playbook_assignments = df.apply(assign_playbook_action, axis=1)
df['reason_code'] = [p[0] for p in playbook_assignments]
df['recommended_action'] = [p[1] for p in playbook_assignments]
df = df.sort_values('opportunity_score', ascending=False).reset_index(drop=True)
df['rank'] = df.index + 1

print('=== PLAYBOOK REASON CODE DISTRIBUTION ===')
print(df['reason_code'].value_counts().to_string())

# Save exported queue CSV
out_path = '../../work/outputs/action_playbook_queue.csv' if os.path.exists('../../work/outputs') else 'work/outputs/action_playbook_queue.csv'
export_cols = ['rank', 'content_id', 'client_id', 'content_type', 'main_intent', 'avg_position', 'position_tier', 'impressions_90d', 'opportunity_score', 'reason_code', 'recommended_action']
df[export_cols].to_csv(out_path, index=False)
print(f'[EXPORTED QUEUE] Saved {len(df):,} rows to {out_path}')


=== PLAYBOOK REASON CODE DISTRIBUTION ===
reason_code
GENERAL_CTR_REVIEW               28331
CTR_BELOW_POSITION_PEER            225
STALE_CONTENT_HIGH_VISIBILITY      204
STRIKING_DISTANCE_POS_11_20         35
[EXPORTED QUEUE] Saved 28,795 rows to work/outputs/action_playbook_queue.csv


## 7. Artifacts the paper embeds

### Visual Figure Generation
Generates and saves visual charts to `work/figures/` for embedding in the research paper.

In [7]:
fig_dir = '../../work/figures' if os.path.exists('../../work/figures') else 'work/figures'
os.makedirs(fig_dir, exist_ok=True)

# 1. Opportunity Score Distribution
plt.figure(figsize=(8, 5))
plt.hist(df['opportunity_score'], bins=50, color='#2563EB', edgecolor='white', alpha=0.85)
plt.axvline(10.0, color='#EF4444', linestyle='--', linewidth=2, label='Action Threshold (>= 10 clicks)')
plt.yscale('log')
plt.title('Model Opportunity Score Distribution (Log Scale)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Missed Clicks (Opportunity Score)', fontsize=10)
plt.ylabel('Page Count (Log Scale)', fontsize=10)
plt.legend()
plt.grid(axis='y', alpha=0.3)
p1 = os.path.join(fig_dir, 'w07_opportunity_score_distribution.png')
plt.savefig(p1, dpi=300, bbox_inches='tight')
plt.close()
print(f'[EXPORTED FIGURE 1] Saved: {p1}')

# 2. Reason Code Distribution
plt.figure(figsize=(8, 5))
rc_counts = df['reason_code'].value_counts()
bars = plt.bar(rc_counts.index, rc_counts.values, color=['#2563EB', '#10B981', '#F59E0B', '#64748B'])
plt.title('Content Action Queue Reason Code Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Reason Code', fontsize=10)
plt.ylabel('Number of Pages', fontsize=10)
plt.xticks(rotation=15, ha='right')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 200, f'{yval:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
p2 = os.path.join(fig_dir, 'w07_reason_code_distribution.png')
plt.savefig(p2, dpi=300, bbox_inches='tight')
plt.close()
print(f'[EXPORTED FIGURE 2] Saved: {p2}')

# 3. Precision@K Comparison Chart
plt.figure(figsize=(8, 5))
k_labels = ['P@10', 'P@20', 'P@50', 'P@100']
rule_prec = [100.0, 100.0, 100.0, 73.0]
model_prec = [90.0, 85.0, 72.0, 54.0]
x = np.arange(len(k_labels))
width = 0.35
plt.bar(x - width/2, rule_prec, width, label='Week 4 Baseline Rule', color='#94A3B8')
plt.bar(x + width/2, model_prec, width, label='Week 6 Honest Model (Grouped)', color='#2563EB')
plt.axhline(1.25, color='#EF4444', linestyle=':', label='Test Base Rate (1.25%)')
plt.title('Precision@K Comparison on Unseen Test Clients (Honest Split)', fontsize=12, fontweight='bold')
plt.ylabel('Precision (%)', fontsize=10)
plt.xticks(x, k_labels)
plt.ylim(0, 115)
plt.legend()
plt.grid(axis='y', alpha=0.3)
p3 = os.path.join(fig_dir, 'w07_precision_at_k_comparison.png')
plt.savefig(p3, dpi=300, bbox_inches='tight')
plt.close()
print(f'[EXPORTED FIGURE 3] Saved: {p3}')


[EXPORTED FIGURE 1] Saved: work/figures\w07_opportunity_score_distribution.png
[EXPORTED FIGURE 2] Saved: work/figures\w07_reason_code_distribution.png


[EXPORTED FIGURE 3] Saved: work/figures\w07_precision_at_k_comparison.png


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to repo under `work/notebooks/capstone.ipynb`